# 🧭 Embeddings: Choosing and Tuning for Your Use Case

## What You'll Learn
- **Part 1 — Theory:** what embeddings actually are (and how to see them cluster)
- **Part 2 — Model choice:** speed/quality trade-offs across common sentence-transformers
- **Part 3 — Domain optimization:** how to evaluate whether an embedding model is right for *your* data

## Prerequisites
Notebook 09 (RAG from Scratch). Embeddings are the retrieval half of RAG — if you skipped 09, the motivation here will land differently.

## Scope
This is a focused tour, not an exhaustive reference. By the end you should know:
1. How to *visualize* an embedding space (t-SNE on real sentence embeddings)
2. How to *benchmark* candidate models on your hardware
3. How to *evaluate* a model's fit for a specific retrieval task

For deeper coverage (embedding arithmetic, quantization, HNSW vs IVF index internals), see the suggested readings at the end.

## Roadmap
| Part | Topic | Time |
|------|-------|------|
| 1 | What embeddings are + t-SNE visualization | 15 min |
| 2 | Speed/quality trade-offs across models | 15 min |
| 3 | Domain-specific evaluation (trace retrieval) | 20 min |


In [ ]:
# Install required libraries
%pip install sentence-transformers scikit-learn matplotlib seaborn

## Part 1: What Are Embeddings? (Theory + Visualization)

**Embeddings = Dense vector representations of meaning**

```
Text: "The cat is sleeping"
       ↓ (Embedding Model)
Vector: [0.23, -0.15, 0.67, ..., 0.42]  # 384 dimensions
```

**Key properties:**
- Similar meanings → nearby vectors
- Vector math works: "king" - "man" + "woman" ≈ "queen"
- Learned from data, not hand-coded

### Experiment 1.1: Visualize Embedding Space

**Goal:** See how embeddings cluster semantically similar sentences

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
import seaborn as sns

def visualize_embedding_space():
    """
    Show how embeddings cluster semantically similar sentences
    """
    
    # Load a real embedding model
    print("Loading sentence-transformers model...")
    model = SentenceTransformer('all-MiniLM-L6-v2')  # 384 dimensions
    
    # Test sentences with known semantic clusters
    sentences = [
        # Cluster 1: Animals
        "The cat sits on the mat",
        "Dogs are loyal companions",
        "The bird flew over the tree",
        "Lions hunt in the savannah",
        
        # Cluster 2: Technology
        "The computer crashed again",
        "Python is a great programming language",
        "Artificial intelligence is the future",
        "My smartphone battery is dead",
        
        # Cluster 3: Food
        "The pizza tastes delicious",
        "I love eating sushi for dinner",
        "Vegetables are healthy for you",
        "The cake was too sweet",
        
        # Cluster 4: Emotions
        "I feel so happy today",
        "This makes me very angry",
        "She is sad about the news",
        "They feel overwhelmed",
    ]
    
    # Generate embeddings (384-dimensional vectors)
    print("Generating embeddings...")
    embeddings = model.encode(sentences)
    print(f"Embedding shape: {embeddings.shape}")  # (16, 384)
    
    # Reduce to 2D for visualization using t-SNE
    print("Reducing to 2D...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=5)
    embeddings_2d = tsne.fit_transform(embeddings)
    
    # Plot
    plt.figure(figsize=(14, 10))
    
    # Color by cluster
    colors = ['red'] * 4 + ['blue'] * 4 + ['green'] * 4 + ['purple'] * 4
    labels = ['Animals'] * 4 + ['Technology'] * 4 + ['Food'] * 4 + ['Emotions'] * 4
    
    for i, (x, y) in enumerate(embeddings_2d):
        plt.scatter(x, y, c=colors[i], s=200, alpha=0.6, edgecolors='black')
        plt.annotate(
            sentences[i], 
            (x, y),
            xytext=(5, 5), 
            textcoords='offset points',
            fontsize=9
        )
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='red', label='Animals'),
        Patch(facecolor='blue', label='Technology'),
        Patch(facecolor='green', label='Food'),
        Patch(facecolor='purple', label='Emotions'),
    ]
    plt.legend(handles=legend_elements, loc='best')
    
    plt.title("Semantic Similarity in Embedding Space\n(384D → 2D via t-SNE)", fontsize=14)
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\nKey Observation:")
    print("  - Sentences about animals cluster together")
    print("  - Technology terms cluster separately")
    print("  - This clustering is LEARNED, not hard-coded!")
    print("  - This is why semantic search works")

if __name__ == "__main__":
    visualize_embedding_space()

## Part 2: Embedding Model Comparison (Empirical)

### The Landscape of Embedding Models

**Common options:**

| Model | Dimensions | Speed | Quality | Use Case |
|-------|------------|-------|---------|----------|
| all-MiniLM-L6-v2 | 384 | Fast | Good | General purpose |
| all-mpnet-base-v2 | 768 | Medium | Better | Higher quality |
| BAAI/bge-large-en-v1.5 | 1024 | Slow | Best | Maximum quality |

**Trade-offs:**
- Larger models = better quality, slower, more memory
- Smaller models = faster, good enough for most cases

### Experiment 2.1: Speed Benchmarks

**Goal:** Measure real-world performance differences

In [ ]:
import time
import numpy as np
from sentence_transformers import SentenceTransformer

def benchmark_embedding_models():
    """
    Compare speed of different embedding models
    """
    
    models = {
        "tiny_fast": "all-MiniLM-L6-v2",        # 384d
        "medium": "all-mpnet-base-v2",          # 768d
        # "large_slow": "BAAI/bge-large-en-v1.5", # 1024d (Commented out to save download time)
    }
    
    # Test corpus (simulate Chatbot traces)
    test_texts = [
        "User expressed anxiety about work deadlines and feeling overwhelmed",
        "Bot suggested breaking down tasks into smaller steps",
        "User felt relieved after discussing the plan",
        "Bot reminded user to take breaks",
        "Previous discussion about coping with stress and pressure",
    ] * 20  # 100 texts total
    
    results = {}
    
    print("=" * 70)
    print("EMBEDDING MODEL SPEED BENCHMARK")
    print("=" * 70)
    print(f"\nTest corpus: {len(test_texts)} texts")
    print()
    
    for name, model_name in models.items():
        print(f"Testing {name} ({model_name})...")
        
        # Load model
        start_load = time.time()
        model = SentenceTransformer(model_name)
        load_time = time.time() - start_load
        
        # Warmup
        model.encode("warmup")
        
        # Benchmark encoding
        start_encode = time.time()
        embeddings = model.encode(test_texts)
        encode_time = time.time() - start_encode
        
        texts_per_sec = len(test_texts) / encode_time
        
        results[name] = {
            "load_time": load_time,
            "encode_time": encode_time,
            "texts_per_sec": texts_per_sec,
            "dim": embeddings.shape[1]
        }
        
        print(f"  - Load time: {load_time:.2f}s")
        print(f"  - Encode time: {encode_time:.4f}s")
        print(f"  - Speed: {texts_per_sec:.1f} texts/sec")
        print(f"  - Dimensions: {embeddings.shape[1]}")
        print()
    
    # Comparison
    print("=" * 70)
    print("COMPARISON")
    print("=" * 70)
    
    baseline = results["tiny_fast"]
    
    for name, res in results.items():
        if name == "tiny_fast":
            continue
            
        speedup = baseline["texts_per_sec"] / res["texts_per_sec"]
        print(f"{name} is {speedup:.1f}x slower than tiny_fast")
    
    print("\n" + "=" * 70)
    print("RECOMMENDATION:")
    print("  For Chatbot trace retrieval:")
    print("    - Start with 'tiny_fast' (384d)")
    print("    - Only upgrade if retrieval quality is insufficient")
    print("    - Speed matters for real-time response")
    print("=" * 70)

if __name__ == "__main__":
    benchmark_embedding_models()

## Part 3: Domain-Specific Optimization

### Why Embedding Choice is a Domain Decision

**General principle:** the *same* embedding model can excel on one task and fail on another. Always evaluate on your own data.


### Experiment 3.1: Evaluate Retrieval Quality

**Goal:** Measure top-1 retrieval accuracy on a small labeled set

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def evaluate_retrieval_model():
    """
    Evaluate an embedding model on a small labeled retrieval set.

    The pattern generalizes to any retrieval task: build a small set of
    (query, expected_doc_id) pairs, embed both sides, and measure top-k
    accuracy. This is the standard offline metric for retrieval quality.
    """

    # A small labeled corpus. Each entry pairs a query with the document
    # that *should* be retrieved for it. In production, you would have
    # hundreds of these gathered from real user feedback or human labeling.
    documents = [
        {
            "id": 0,
            "doc": "Workplace stress often improves when tasks are broken into smaller pieces. Try time-blocking.",
            "query": "I am feeling overwhelmed by deadlines",
        },
        {
            "id": 1,
            "doc": "Python is beginner-friendly. Start with the official tutorial, then build small projects.",
            "query": "How do I start learning Python?",
        },
        {
            "id": 2,
            "doc": "Memory retrieval walks back through stored conversation summaries to find prior context.",
            "query": "What did we talk about last week?",
        },
    ]

    # In a real evaluation set the *test queries* differ from the example
    # queries that produced each document. Here we hold out phrasings as
    # held-out test cases so the model must generalize rather than memorize.
    test_queries = [
        ("I'm drowning in work and can't keep up", 0),
        ("What's a good way to begin programming?", 1),
        ("Remind me what we discussed previously", 2),
    ]

    models_to_test = {
        "tiny_384d": "all-MiniLM-L6-v2",
        # "medium_768d": "all-mpnet-base-v2",  # uncomment to compare
    }

    print("=" * 70)
    print("RETRIEVAL QUALITY EVALUATION")
    print("=" * 70)
    print(f"Corpus: {len(documents)} documents, {len(test_queries)} held-out queries")
    print()

    for model_name, model_path in models_to_test.items():
        print(f"--- Testing {model_name} ({model_path}) ---")
        model = SentenceTransformer(model_path)

        # Embed the documents and the test queries once each.
        doc_embeddings = model.encode([d["doc"] for d in documents])
        query_embeddings = model.encode([q for q, _ in test_queries])

        # Top-1 accuracy: for each query, the highest-similarity doc must match the expected id.
        sim = cosine_similarity(query_embeddings, doc_embeddings)
        top1 = sim.argmax(axis=1)
        correct = sum(int(top1[i] == expected) for i, (_, expected) in enumerate(test_queries))
        accuracy = correct / len(test_queries)
        print(f"  Top-1 accuracy: {accuracy:.0%} ({correct}/{len(test_queries)})")

        # Print a confusion-style view to make the failures visible.
        for i, (q, expected) in enumerate(test_queries):
            status = "OK" if top1[i] == expected else "MISS"
            print(f"    [{status}] '{q}' -> doc {top1[i]} (expected {expected})")
        print()

if __name__ == "__main__":
    evaluate_retrieval_model()


## Summary & Key Takeaways

### What You Learned

1. **Embeddings = Semantic meaning in vector space**
   - Not just word overlap
   - Capture similarity, synonyms, paraphrases

2. **Model Selection Trade-offs**
   - 384d: Fast, good enough for most cases
   - 1024d: 3x slower, ~15% better

3. **Chatbot Specific**
   - Trace retrieval depends on embeddings
   - Upgrade only if retrieval is bottleneck

### Decision Framework

**Start here:**
- Use `all-MiniLM-L6-v2` (384d)
- Fast, proven, good baseline

**Upgrade if:**
- Retrieval quality < 80%
- Queries returning wrong traces
- Speed not critical